# <center>Laboratorio 9: Benchmark de Carga y Modelos con Spotify 🎵</center>

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos</strong></center>

---

### Cuerpo Docente

- Profesores: Pablo Badilla y Diego Cortez
- Auxiliares: Valentina Rojas y Melanie Peña
- Ayudantes: Javiera Arévalo, Tamara Carrasco e Ignacio Reyes

### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Agustin Eduardo Gonzalez Hidalgo
- Nombre de alumno 2: Vicente Ignacio Thiele Muñoz

---

### Reglas

- **Grupos de 2 personas**
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Prohibido copiar.
- Uso de LLM (Copilot, Claude, Cursor, etc.) restringido a consultas, documentación y corrección de errores.

# Temas a tratar

- Lectura eficiente de datos en formato Parquet.
- Optimización del uso de memoria mediante conversión de tipos de datos.
- Paralelización de operaciones I/O con `ThreadPoolExecutor`.
- Comparación de implementaciones de predicción: Python, NumPy, Numba, pandas y Polars.
- Entrenamiento de modelos con RandomForestRegressor y efecto de `n_jobs`.
- Orquestación de pipelines de datos con Apache Airflow.

# Objetivos principales del laboratorio

- Cargar datos de canciones de Spotify desde archivos Parquet y optimizar su representación en memoria.
- Comparar el tiempo de lectura de archivos en serie vs. en paralelo.
- Analizar el impacto de distintas implementaciones (Python puro, NumPy, Numba, pandas, Polars) en el tiempo de predicción de un modelo lineal.
- Entrenar un RandomForestRegressor que prediga la valencia de canciones, comparando el efecto de la paralelización del entrenamiento.
- Orquestar el pipeline completo (carga + entrenamiento) usando Apache Airflow.

> Instalamos e importamos las librerías necesarias 🎸

In [1]:
!uv add pandas pyarrow lightgbm scikit-learn plotly apache-airflow polars numba

Resolved 263 packages in 17ms
Audited 253 packages in 40ms


In [2]:
import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from pathlib import Path

import numba
import numpy as np
import pandas as pd
import plotly.express as px
import polars as pl
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split

DATA_DIR = Path("data")

# 1. Carga y Optimización de Datos con Parquet

Los datos que usaremos en este laboratorio corresponden a un dataset de canciones de Spotify almacenado en **20 archivos Parquet** (`batch_01.parquet` … `batch_20.parquet`), con un total de 200 000 canciones y 24 columnas que incluyen características de audio, metadatos y la letra completa de cada canción.

A continuación trabajaremos en dos aspectos fundamentales de la carga de datos en la práctica:
1. **Optimizar el uso de memoria** ajustando los tipos de datos de las columnas.
2. **Reducir el tiempo de carga** paralelizando la lectura de archivos.

## 1.1 Exploración y Optimización de Tipos de Datos [1 Punto]

Cuando cargamos datos con pandas, los tipos inferidos por defecto no siempre son los más eficientes. Por ejemplo, un entero que siempre cabe en 16 bits se almacena por defecto como `int64` (64 bits), usando 4 veces más memoria de la necesaria. Lo mismo ocurre con flotantes y con columnas categóricas almacenadas como strings.

**Código dado** — funciones de carga:

In [3]:
def load_batch(path: str) -> pd.DataFrame:
    """Lee un único archivo Parquet y retorna un DataFrame."""
    return pd.read_parquet(path)


def load_all_serial(data_dir: Path, n_batches: int | None = None) -> pd.DataFrame:
    """Lee todos los archivos Parquet de data_dir en serie y los concatena."""
    paths = sorted(data_dir.glob("*.parquet"))
    if n_batches is not None:
        paths = paths[:n_batches]
    return pd.concat([load_batch(str(p)) for p in paths], ignore_index=True)

**TO-DO [0.3 Puntos]:**
- [X] Ejecutar `load_all_serial` sobre todos los batches y explorar el DataFrame resultante (`.dtypes`, `.memory_usage(deep=True)`).
- [X] Aplicar las siguientes conversiones a un nuevo DataFrame (copia del originalmente cargado `df_opt`):
  - `float64` → `float32`: columnas de audio features (`danceability`, `energy`, `loudness`, `speechiness`, `acousticness`, `instrumentalness`, `liveness`, `valence`, `tempo`, `avg_artist_popularity`).
  - `int64` → `int16`: columnas `key`, `mode`.
  - `int64` → `int32`: columnas `year`, `popularity`, `duration_ms`, `total_artist_followers`.
- [X] Comparar el uso de memoria antes y después con un gráfico de barras usando Plotly (código dado).

In [ ]:
df = load_all_serial(DATA_DIR)

print(df.dtypes)
print("\nUso de memoria por columna (MiB):")
print((df.memory_usage(deep=True) / 1024**2).sort_values(ascending=False))

df_opt = df.copy()

# float64 → float32: audio features
float32_cols = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "avg_artist_popularity",
]
df_opt[float32_cols] = df_opt[float32_cols].astype("float32")

# int64 → int16: columnas de rango pequeño
df_opt[["key", "mode"]] = df_opt[["key", "mode"]].astype("int16")

# int64 → int32
df_opt[["year", "popularity", "duration_ms", "total_artist_followers"]] = df_opt[
    ["year", "popularity", "duration_ms", "total_artist_followers"]
].astype("int32")

# Compara el uso de memoria antes y después con un gráfico de barras
mem_before = df.memory_usage(deep=True).sum() / 1024**2
mem_after = df_opt.memory_usage(deep=True).sum() / 1024**2

px.bar(
    x=["Antes", "Después"],
    y=[mem_before, mem_after],
    labels={"x": "Estado", "y": "Uso de Memoria (MiB)"},
    title=f"Uso de memoria: {mem_before:.1f} MiB → {mem_after:.1f} MiB ({(1 - mem_after / mem_before) * 100:.1f}% reducción)",
).show()

id                            str
name                          str
album_name                    str
artists                    object
danceability              float64
energy                    float64
key                         int64
loudness                  float64
mode                        int64
speechiness               float64
acousticness              float64
instrumentalness          float64
liveness                  float64
valence                   float64
tempo                     float64
duration_ms                 int64
lyrics                        str
year                        int64
genre                         str
popularity                  int64
total_artist_followers      int64
avg_artist_popularity     float64
artist_ids                 object
niche_genres               object
dtype: object

Uso de memoria por columna (MiB):
lyrics                    246.823409
niche_genres               22.888184
artist_ids                 22.888184
artists                 

### Preguntas [0.7 Puntos]

1. ¿Qué es el formato **Parquet**? ¿Qué ventajas tiene sobre CSV para datos analíticos? ¿Qué es *columnar storage* y por qué acelera las consultas que solo leen algunas columnas?
2. ¿Qué es **Apache Arrow**? ¿Cómo se relaciona con Parquet y con pandas internamente? ¿Qué ganas al usar `pd.read_parquet` en vez de `pd.read_csv`?
3. ¿Por qué existe `float32` si `float64` es más preciso? ¿En qué contextos esa pérdida de precisión es irrelevante?
4. ¿Cuándo **no** conviene reducir la precisión de un tipo numérico? ¿Qué riesgos concretos existen?
5. ¿Existe alguna alternativa a pandas para trabajar con estos datos de forma más eficiente en memoria? (menciona al menos dos)
6. ¿Cuánto se redujo el uso de memoria en total (en MiB y en %)? ¿Era esperable ese resultado? ¿Por qué no se redujo tanto como podría esperarse?
7. ¿Qué pasaría si intentaras reducir `valence` a `float16`? ¿Qué riesgo existiría para el modelo entrenado en la sección 2?

**Respuestas:**

**1.**

Parquet es un formato de almacenamiento de datos en columnas, diseñado para análisis eficiente. A diferencia de CSV, que almacena los datos fila por fila como texto plano, parquet agrupa los valores de cada columna de forma contigua en disco y aplica compresión específica por tipo. Las principales ventajas sobre CSV son:
- **Compresión**: Agrupa valores similares juntos, esto permite que se compriman mucho mejor. CSV no comprime.
- **Tipado estricto**: Cada columna tiene un tipo definido en el esquema, evitando conversiones al leer.
- **Lectura selectiva de columnas**: Como los datos de cada columna están contiguos, se puede leer solo las columnas necesarias sin leer todas las filas. En CSV hay que parsear toda la fila siempre.
- **Velocidad en consultas analíticas**: Las operaciones `SELECT col_a, col_b FROM ...` se benefician directamente porque solo se leen los bytes de esas columnas.

*Columnar storage*: En vez de guardar `[fila1_col1, fila1_col2, fila2_col1, fila2_col2, ...]`, guarda `[col1_fila1, col1_fila2, ..., col2_fila1, col2_fila2, ...]`. Esto permite leer solo las columnas que una consulta necesita, reduciendo drásticamente las I/Os.

**2.**

Apache Arrow es un estándar de formato de datos **en memoria**, diseñado para ser eficiente con CPU modernas, datos contiguos en columnas, sin copias innecesarias, acceso directo a través de *zero-copy*.

- Arrow define el formato en memoria, Parquet define el formato en disco. Leer un Parquet con PyArrow traduce los datos desde el formato comprimido en disco hacia la representación Arrow en RAM, sin pasar por Python puro.
- `pd.read_parquet` usa PyArrow. Al leer, PyArrow carga los datos en formato Arrow y luego los convierte a DataFrames pandas. Esto es mucho más rápido que `pd.read_csv` porque no hay que parsear texto, los tipos ya vienen definidos en el esquema Parquet, PyArrow usa código C++ vectorizado para la descompresión y conversión.

**3.**

`float32` ocupa 4 bytes frente a los 8 bytes de `float64`, exactamente la mitad de memoria. Esa perdida de precisión no importa en los siguientes casos:
- Se trabaja con **modelos de machine learning**: La mayoría de modelos no necesitan más de 6-7 dígitos significativos. Los pesos entrenados con `float32` son prácticamente iguales en calidad a los de `float64`.
- Se trabaja en **GPU**: Las GPUs actuales tienen el doble de rendimiento en `float32` vs `float64`, y los frameworks de deep learning como PyTorch o TensorFlow operan en `float32` por defecto.

**4.**

No conviene reducir la precisión cuando:
- **Se acumulan errores de redondeo**: En sumas de millones de valores pequeños, los errores de `float32` se propagan, esto se puede notar porque `float32` tiene aproximadamente 7 dígitos significativos, si se suman millones de valores, el error acumulado puede ser significativo.
- **Se requiere reproducibilidad exacta**: Operaciones con `float32` no son deterministas entre plataformas de la misma manera que `float64`.

**5.**

Sí, hay alternativas:

- **Polars**: Librería implementada en Rust con modelo de ejecución lazy y paralelismo automático. Usa Apache Arrow como formato interno, lo que la hace más eficiente en memoria y velocidad que pandas para la mayoría de operaciones. Es de código abierto.

- **DuckDB**: Motor SQL analítico embebido que puede leer Parquet directamente sin cargar todo en memoria. Ideal para consultas SQL sobre archivos Parquet de forma eficiente. Permite hacer `SELECT` sobre archivos sin cargar el DataFrame completo.

**6.**

La reducción de memoria es de 358.7 a 345.7 MiB y en porcentaje esto se traduce en un 3.6% de reducción.

Esperabamos un poco más de reducción, esperabamos un valor mucho mayor.

Creemos que la reducción no fue tanto porque columnas como `track_name`, `artist_name`, `genre`, `lyrics`, no fueron "optimizadas" y estas por ser strings largos, sobretodo `lyrics`, que es la letra completa de la canción, suelen pesar más que las demás, que si fueron optimizadas, en terminos de memoria.

**7.**

`float16` tiene solo 3 dígitos significativos y rango limitado. Para `valence` (rango 0–1), los valores se representarían con muy poca precisión. La resolución mínima de `float16` en el rango [0,1] es de aproximadamente 0.001, lo que significa que valores pordrian redondearse erroneamente.

`valence` es la **variable objetivo** de la sección 2. Un error de representación en esta implica que el modelo aprende sobre una señal ruidosa/distorsionada, lo que puede aumentar el RMSE de forma artificial. En regresión, la precisión del target es crítica, si el target tiene ruido de ±0.001 por cuantización, el RMSE mínimo alcanzable quedará limitado a ese nivel de ruido. Para un modelo que busca predecir `valence` con precisión, esto representaría una degradación measurable del rendimiento.

In [5]:
# **IMPORTANTE**: Una vez contestada la pregunta, ejecutar esta celda para liberar memoria.
df_opt = None

## 1.2 Lectura en Serie vs. Paralelo [1 Punto]

Cuando se trabaja con múltiples archivos, la lectura **en paralelo** puede reducir el tiempo total al aprovechar que la espera de I/O (disco/red) no bloquea al procesador. En Python, la clase `ThreadPoolExecutor` del módulo `concurrent.futures` permite lanzar múltiples hilos para ejecutar operaciones de forma concurrente.

**TO-DO: [0.3 Puntos]**
- [X] Implementar `load_all_parallel` usando `ThreadPoolExecutor`.
- [ ] Medir con `%timeit` ambas versiones sobre todos los batches.
- [X] Generar un gráfico de línea (Plotly) con los tiempos para 2, 4, 6, …, 20 archivos, con series `Serial` y `Paralelo`.

In [8]:
def load_all_parallel(data_dir: Path, n_batches: int | None = None) -> pd.DataFrame:
    paths = sorted(data_dir.glob("*.parquet"))
    if n_batches is not None:
        paths = paths[:n_batches]
    with ThreadPoolExecutor() as executor:
        dfs = list(executor.map(load_batch, [str(p) for p in paths]))
    return pd.concat(dfs, ignore_index=True)

**Benchmark:** mide tiempos para 2, 4, 6, ..., 20 archivos y grafica


In [7]:
@dataclass
class ReadMeasurement:
    n_files: int
    time_sec: float
    version: str


measurements: list[ReadMeasurement] = []

for n in range(2, 21):
    t0 = time.perf_counter()
    load_all_serial(DATA_DIR, n_batches=n)
    measurements.append(ReadMeasurement(n, time.perf_counter() - t0, "Serial"))

    t0 = time.perf_counter()
    load_all_parallel(DATA_DIR, n_batches=n)
    measurements.append(ReadMeasurement(n, time.perf_counter() - t0, "Paralelo"))

df_times = pd.DataFrame(measurements)
px.line(
    df_times,
    x="n_files",
    y="time_sec",
    color="version",
    markers=True,
    title="Tiempo de lectura: Serial vs Paralelo",
    labels={"n_files": "Número de archivos", "time_sec": "Tiempo (s)"},
).show()

### Preguntas  [0.7 Puntos]

1. ¿Qué significa que una operación sea **I/O-bound** vs **CPU-bound**? ¿A cuál categoría pertenece la lectura de archivos desde disco?
2. ¿Qué es el **GIL** (*Global Interpreter Lock*) de CPython? ¿Por qué existe? ¿Qué problema resuelve y qué limitación introduce?
3. ¿Por qué usamos Python si tiene el GIL? ¿Qué ganamos al usarlo como lenguaje de *pegamento* entre librerías de alto rendimiento (NumPy, Arrow, PyTorch…)?
4. ¿Cuándo conviene usar `ThreadPoolExecutor` vs `ProcessPoolExecutor`? ¿Cuál usarías si la operación fuera puramente CPU-bound?
5. ¿Qué overhead introduce crear un pool de threads? ¿Qué pasaría si los archivos fueran muy pequeños (p.ej. 1 KB cada uno)?
6. ¿Se observó mejora con la lectura paralela? ¿A partir de cuántos archivos empieza a ser notable?
7. ¿Por qué el speedup obtenido **no es igual** al número de threads disponibles? ¿Qué factores lo limitan?

**Respuestas:**

**1.**

Una operación es **CPU-bound** cuando la velocidad de ejecución de una tarea depende de la capacidad de cómputo del procesador. Una operación es **I/O-bound** cuando la velocidad de ejecución depende de la velocidad de transferencia de datos. La lectura de archivos desde disco es claramente **I/O-bound**, el CPU lanza la solicitud de lectura y luego espera a que el disco entregue los bytes. Durante esa espera el CPU podría estar ocupado haciendo otro trabajo.

**2.**

El **GIL** es un mutex en CPython que asegura que solo **un hilo Python ejecuta bytecode a la vez**, incluso en sistemas con múltiples núcleos.

Existe por razones históricas de simplicidad: CPython gestiona memoria con conteo de referencias, y ese conteo no es thread-safe. El GIL protege la integridad del conteo de referencias sin necesitar un lock por objeto. Simplifica enormemente la implementación del intérprete y la interoperabilidad con librerías C.

Este principalmente evita condiciones de carrera en la gestión de memoria interna de Python.

Como limitación tenemos que los threads de Python **no pueden paralelizar trabajo CPU-bound** real. Si dos threads ejecutan código Python puro en paralelo, el GIL fuerza que se alternen, sin ganar velocidad respecto a un solo thread. La paralelización CPU-bound real en Python requiere procesos separados (`ProcessPoolExecutor`, `multiprocessing`).

**3.**

Los beneficios de Python son:
- Tiene una sintaxis clara y productiva para orquestar tareas complejas.
- Las librerías críticas (NumPy, Pandas, PyArrow, PyTorch, scikit-learn) están implementadas en C/C++/Rust y **liberan el GIL** durante su ejecución, permitiendo paralelismo real dentro de sus operaciones.
- El ecosistema científico de Python (Jupyter, matplotlib, scikit-learn) ya está muy desarrollado en el área productiva.

En la práctica, en código de data science, gran parte del tiempo de CPU lo ocupa código C/C++, no el intérprete Python. El GIL solo es un problema cuando se escribe código Python puro en loops intensivos, que es exactamente lo que se evita con vectorización y librerías optimizadas.

**4.**

- **`ThreadPoolExecutor`**: Ideal para operaciones **I/O-bound**. Los threads comparten memoria, tienen bajo overhead de creación, y el GIL se libera durante operaciones I/O. Perfecto para leer múltiples archivos, hacer requests HTTP concurrentes, consultar bases de datos.

- **`ProcessPoolExecutor`**: Ideal para operaciones **CPU-bound** que ejecutan código Python puro. Cada proceso tiene su propio intérprete Python con su propio GIL, permitiendo paralelismo real en CPU. El overhead es mayor y la comunicación entre procesos requiere serialización. Útil para aplicar funciones Python puras sobre chunks de datos.

Para la lectura de archivos Parquet, `ThreadPoolExecutor` es la elección correcta y más eficiente.

**5.**

Crear un `ThreadPoolExecutor` tiene un overhead fijo: Inicialización del pool, creación de los threads del sistema operativo, sincronización inicial. Este overhead es típicamente de algunos milisegundos a decenas de milisegundos. Si los archivos fueran muy pequeños (1 KB cada uno), el tiempo de lectura de cada archivo sería de microsegundos, menor que el overhead del pool. En ese caso, la versión serial sería más rápida que la paralela porque el costo de administrar los threads superaría el beneficio de concurrencia. La paralelización solo tiene sentido cuando el trabajo por unidad es significativamente mayor que el overhead del pool.

**6.**

Sí, se observa mejora con la lectura paralela, aunque el speedup no es lineal con el número de threads. En el gráfico, la versión paralela tiende a ser más rápida a partir de 10 archivos aproximadamente, cuando el tiempo de I/O real empieza a justificar el overhead del pool. Para menos archivos, el overhead del pool puede dominar y hacer que la versión paralela sea igual o ligeramente más lenta. A medida que aumenta el número de archivos, la ventaja de la lectura paralela se vuelve más evidente, ya que múltiples archivos se leen concurrentemente mientras el CPU procesa los ya leídos.

**7.**

El speedup real siempre es menor que el número de threads por varias razones:

- **Cuello de botella en disco**: Todos los threads compiten por el mismo bus de almacenamiento. Si el disco solo puede servir X MB/s, tener más threads no aumenta la velocidad más allá de esa limitación física.
- **Overhead del pool**: Creación de threads, sincronización, cola de tareas.
- **Caché del OS**: La primera lectura puede ser más lenta, lecturas posteriores pueden beneficiarse del *page cache* del sistema operativo, haciendo que la versión serial también sea rápida en ejecuciones repetidas.

# 2. Predicción de Valencia

La columna `valence` de Spotify mide el **positivismo musical** de una canción: valores cercanos a 1 indican canciones alegres y eufóricas, mientras que valores cercanos a 0 corresponden a canciones tristes o melancólicas. En esta sección analizaremos distintas formas de realizar predicciones con un modelo de regresión lineal ya entrenado, y luego entrenaremos un modelo más complejo.

## 2.1 Regresión Lineal a Mano [1.5 Puntos]

Antes de entrenar un modelo completo, veremos cómo **la elección de implementación** afecta drásticamente el rendimiento de predicción. Usaremos un modelo de regresión lineal pre-entrenado cuyos coeficientes ya están dados, e implementaremos la predicción usando cinco enfoques distintos: Python puro, NumPy, Numba (JIT), pandas y Polars.

**Código dado — carga de datos y parámetros del modelo:**

In [9]:
# Carga de datos y preparación del split
df_train = load_all_serial(DATA_DIR, n_batches=20)

PARAM_COLS = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "tempo",
    "duration_ms",
    "year",
]

X = df_train[PARAM_COLS + ["key", "mode", "genre"]]
y = df_train["valence"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [10]:
# Parámetros del modelo lineal pre-entrenado (dados)
params = {
    "danceability": 0.7718203106411208,
    "energy": 0.4252134896942928,
    "loudness": -0.008319917439312445,
    "speechiness": -0.24543273088867107,
    "acousticness": 0.10440236785191129,
    "instrumentalness": -0.11203723673701874,
    "liveness": 0.023790522969698424,
    "tempo": 0.0007885690378158087,
    "duration_ms": -4.31739613602265e-07,
    "year": -0.0036043842721972985,
}
intercept = 6.948154825159983

params_vals = list(params.values())
params_arr = np.array(params_vals, dtype=np.float32)

**Código dado — las 5 implementaciones de predicción:**

Analiza cómo cada implementación aborda el mismo problema y presta atención a las diferencias en legibilidad, concisión y (como verás en el benchmark) rendimiento.

In [ ]:
def linear_regression_predict(X: np.ndarray, params: list[float]) -> list[float]:
    """Predicción con loop Python puro."""
    preds = []
    for row in X:
        val = intercept
        for j, w in enumerate(params):
            val += row[j] * w
        preds.append(val)
    return preds


def linear_regression_predict_numpy(X: np.ndarray, params: list[float]) -> np.ndarray:
    """Predicción vectorizada con NumPy."""
    return (X * np.array(params)).sum(axis=1) + intercept


@numba.njit
def linear_regression_predict_numba(X: np.ndarray, params: np.ndarray) -> np.ndarray:
    """Predicción con Numba JIT (loop compilado a código máquina)."""
    n = X.shape[0]
    preds = np.empty(n)
    for i in range(n):
        val = intercept
        for j in range(len(params)):
            val += X[i, j] * params[j]
        preds[i] = val
    return preds


def linear_regression_predict_pandas(X: pd.DataFrame, params: list[float]) -> pd.Series:
    """Predicción vectorizada con pandas (dot product)."""
    return X.dot(pd.Series(params, index=X.columns)) + intercept


def linear_regression_predict_polars(X: pl.DataFrame, params: list[float]) -> pl.Series:
    """Predicción vectorizada con Polars (expresiones lazy)."""
    weights = dict(zip(X.columns, params, strict=False))
    expr = pl.lit(intercept)
    for col, w in weights.items():
        expr = expr + pl.col(col) * w
    return X.select(expr.alias("pred"))["pred"]

**Código dado — Benchmark de las 5 implementaciones:**

In [ ]:
@dataclass
class TimeMeasurement:
    time_took: float
    iteration: int
    version: str


time_measurements: list[TimeMeasurement] = []

ranges = [10, 50, 100, 250, 500, 750, 1000, *range(1001, len(X_test) + 1, 1000)]

for it in ranges:
    X_np = X_test[PARAM_COLS].iloc[:it].to_numpy(dtype=np.float32)
    X_pd = X_test[PARAM_COLS].iloc[:it]
    X_pl = pl.from_pandas(X_pd)

    for name, fn, args in [
        ("Python", linear_regression_predict, (X_np, params_vals)),
        ("NumPy", linear_regression_predict_numpy, (X_np, params_vals)),
        ("Numba-JIT", linear_regression_predict_numba, (X_np, params_arr)),
        ("Pandas", linear_regression_predict_pandas, (X_pd, params_vals)),
        ("Polars", linear_regression_predict_polars, (X_pl, params_vals)),
    ]:
        t0 = time.perf_counter()
        fn(*args)
        time_measurements.append(TimeMeasurement(time.perf_counter() - t0, it, name))

df_bench = pd.DataFrame(time_measurements)

# Gráfico 1: tiempos absolutos
px.line(
    df_bench,
    x="iteration",
    y="time_took",
    color="version",
    markers=True,
    title="Tiempos de predicción según implementación",
    labels={"iteration": "Número de filas", "time_took": "Tiempo (s)"},
).show()

# Gráfico 2: tiempos absolutos (en log)
px.line(
    df_bench,
    x="iteration",
    y="time_took",
    color="version",
    markers=True,
    title="Tiempos de predicción según implementación (en escala logarítmica)",
    labels={"iteration": "Número de filas", "time_took": "Tiempo (s)"},
    log_y=True,
).show()

# Gráfico 3: speedup relativo respecto a Python puro
pivot = df_bench.pivot(index="iteration", columns="version", values="time_took")
for col in ["NumPy", "Numba-JIT", "Pandas", "Polars"]:
    pivot[col] = pivot["Python"] / pivot[col]
pivot["Python"] = 1.0

melted = pivot.reset_index().melt(
    id_vars=["iteration"],
    value_vars=["Python", "NumPy", "Numba-JIT", "Pandas", "Polars"],
    value_name="speedup",
)
px.line(
    melted,
    x="iteration",
    y="speedup",
    color="version",
    markers=True,
    title="Speedup relativo respecto a Python puro",
    labels={"iteration": "Número de filas", "speedup": "Speedup (×)"},
).show()

### Preguntas [1.5 Puntos]

  1. ¿Qué es la vectorización en NumPy? ¿Cómo puede ejecutar operaciones sobre arrays sin loops de Python explícitos?
  2. ¿Qué es JIT (Just-In-Time compilation)? ¿Qué hace el decorador @numba.njit? ¿Qué significa el modo nopython?
  3. ¿Por qué Numba es más lento en la primera ejecución? ¿Qué es el warm-up de JIT y cómo lo manejamos en el benchmark?
  4. ¿Qué es Polars y cuáles son sus principales características como librería de datos? ¿Para qué escenarios fue diseñada y por qué ha ganado popularidad como alternativa a pandas?
  5. ¿En qué se diferencia Polars de pandas a nivel de implementación (lenguaje, modelo de ejecución, manejo de memoria)?
  6. ¿Por qué pandas puede ser más lento que NumPy aun usando operaciones vectorizadas internamente?
  7. ¿Qué son las instrucciones SIMD (Single Instruction Multiple Data)? ¿Cómo contribuyen a la aceleración de NumPy y Polars?
  8. ¿Cuándo conviene usar Numba sobre NumPy? ¿Y Polars sobre pandas para operaciones numéricas?
  9. ¿Cuál implementación fue la más rápida en tu medición? ¿Era esperable ese resultado?
  10. ¿Se observa diferencia notable entre pandas y NumPy? ¿Por qué pandas puede ser más lento o más rápido?
  11. ¿A partir de cuántas filas empieza a ser evidente la ventaja de NumPy/Numba sobre Python puro?
  12. ¿Polars fue más eficiente que pandas en tu medición? Verifica la versión de pandas instalada (pd.__version__) y comenta si crees que la versión influye en el resultado.
  13. ¿Por qué Numba puede igualar o superar a NumPy para loops numéricos simples?
  14. El benchmark excluye el costo de convertir datos a NumPy/Polars (la conversión ocurre fuera del timing). ¿Cómo cambiaría el resultado si incluyeras ese costo? ¿En qué escenarios de producción ese costo no existiría?
  15.  Si tuvieras que realizar esta predicción sobre 100 millones de filas en un servidor de producción, ¿qué implementación elegirías y por qué? ¿Cambiaría tu respuesta si dispusieras de una GPU?

**Respuestas:**

**1.**

La vectorización en NumPy significa expresar operaciones sobre arrays completos en vez de iterar elemento por elemento en Python. Por ejemplo, en vez de un `for i in range(n): result[i] = a[i] * b[i]`, se escribe `result = a * b`. Internamente, NumPy implementa esa operación en C/Fortran, iterando sobre los elementos en código nativo compilado, que es mucho más rápido que el loop equivalente en Python puro.

**2.**

**JIT** es una técnica donde el código no se compila por adelantado ni se interpreta línea por línea, sino que se compila a código máquina nativo la primera vez que se llama la función, con los tipos reales de los argumentos. En llamadas posteriores, se ejecuta el código nativo ya compilado.

El decorador **`@numba.njit`** le dice a Numba que compile esa función usando JIT en modo "nopython". En este modo, Numba compila toda la función a código máquina sin usar el intérprete Python en ningún punto, si encuentra algo que no sabe compilar, lanza un error en vez de caer silenciosamente al modo Python. Esto garantiza que la función compilada sea rápida, sin partes interpretadas escondidas.

**3.**

La primera llamada a una función `@numba.njit` activa la compilación JIT: Numba analiza los tipos de los argumentos, genera código Intermediate Representation (IR), lo optimiza y finalmente lo compila a código máquina. Este proceso puede tardar segundos.

El **warm-up** de JIT es precisamente este costo de compilación en la primera llamada. En el benchmark del laboratorio, esto se maneja ejecutando el benchmark sobre conjuntos de datos de tamaño creciente. La primera iteración con 10 filas paga el costo de compilación, las iteraciones siguientes ejecutan directamente el código ya compilado. En producción se suele hacer un *warm-up* explícito con datos dummy antes de empezar a medir tiempos reales.

**4.**

**Polars** es una librería de DataFrames implementada en Rust, lanzada en 2021. Fue diseñada específicamente para:
- Análisis de datos en memoria en un solo nodo, con énfasis en velocidad y eficiencia de memoria.
- Datasets medianos a grandes que caben en RAM.
- Workloads donde pandas se vuelve demasiado lento o consume demasiada memoria.

Ha ganado popularidad por ser consistentemente más rápida que pandas en benchmarks de transformaciones, API expresiva con ejecución lazy y plan de query optimizado, modelo de datos Arrow nativo sin copias, paralelismo automático sin necesidad de configurar threads manualmente.

**5.**

- **Lenguaje**: pandas usa C/CPython sobre Python como lenguaje, en cambio Polars usa Rust como lenguaje.
- **Modelo de ejecución**: pandas es eager, mientras que Polars es muzcla de lazy con eager, en donde el modo lazy optimiza el plan antes de ejecutar el código.
- **Manejo de memoria**: pandas tiene mayor overhead por el boxing de Python, por otro lado, Polars no tiene boxing, usa los tipos nativos de Arrow que sean más eficientes.

**6.**

Pandas añade varias capas de abstracción sobre NumPy:
- **Overhead de Python**: Creación de objetos `pd.Series`, `pd.DataFrame`, metadata de dtypes, validación de argumentos.
- **Boxing de tipos**: Algunas operaciones de pandas convierten datos a tipos Python intermedios antes de procesar.
- **NA handling**: pandas verifica o maneja NaN en muchas operaciones.

En cambio, `np.dot(X, w)` opera directamente sobre el buffer de memoria contiguo sin ninguna de estas capas. La diferencia es más pronunciada en operaciones pequeñas donde el overhead relativo es mayor.

**7.**

**SIMD** son instrucciones de la CPU que aplican la misma operación a múltiples datos simultáneamente en un solo ciclo de reloj.

NumPy, al estar implementado en C, puede aprovechar estas instrucciones automáticamente. Polars, implementado en Rust, también genera código SIMD a través del compilador LLVM y la librería Arrow.

En una operación de multiplicación de arrays de floats, el CPU puede procesar 8 elementos en el mismo tiempo que procesaría 1, lo que da un speedup teórico solo por SIMD, independientemente de otros factores. Lo cual contribuye a la aceleración significativamente.

**8.**

**Numba sobre NumPy**: Cuando la operación no se puede expresar eficientemente con operaciones vectorizadas de NumPy, es decir, cuando hay loops con dependencias entre iteraciones, accesos no contiguos a memoria, lógica condicional compleja elemento a elemento, o algoritmos recursivos. Numba compila el loop directamente a código máquina, siendo equivalente o superior a NumPy en esos casos. También es valioso cuando se quiere usar CUDA (GPU) a través de `@numba.cuda.jit`.

**Polars sobre pandas**: Cuando se trabaja con DataFrames grandes, se hacen múltiples operaciones de transformación en cadena, en donde el plan lazy de Polars puede optimizar el orden, cuando se necesita máximo rendimiento en operaciones groupby/join, o cuando la eficiencia de memoria es crítica.

**9.**

En la medición, **Numba-JIT** resulta ser la implementación más rápida para conjuntos de datos grandes. Esto era esperable, ya que se ejecuta código nativo compilado con posibles optimizaciones SIMD, sin overhead del intérprete Python ni de estructuras de datos de alto nivel. Para tamaños pequeños, el warm-up de Numba lo hace más lento inicialmente, pero una vez compilado supera a NumPy en loops simples.

**10.**

Se observa una diferencia, pero no es tan notable, Numpy suele ser mejor, pero no por tanto. A medida que aumentan las filas se van acercando cada vez más. La razón principal de esta diferencia es el overhead de pandas mencionado en la pregunta 6.

En algunos escenarios pandas puede ser comparable a NumPy si la operación interna es simplemente un `np.dot` sin overhead significativo. En este benchmark, `X.dot(pd.Series(params, index=X.columns))` en pandas requiere crear un objeto `pd.Series`, alinear índices, y luego delegar a NumPy internamente, lo que lo hace ligeramente más lento.

**11.**

La ventaja de NumPy y Numba sobre Python puro se vuelve evidente a partir de aproximadamente 250 filas en adelante. Para tamaños menores, el overhead de crear arrays NumPy, verificar tipos, y la compilación JIT puede dominar.

**12.**

Fue un poco más eficiente, pero no es mucha la diferencia, esto se puede deber a que Polars puede ser comparable o ligeramente más rápido que pandas para conjuntos grandes gracias a su motor Rust y Arrow. Sin embargo, para operaciones simples de dot product sobre pocas columnas numéricas, la diferencia no es tan pronunciada como en operaciones de agrupación o join.

`pd.__version__` en este entorno es **3.0.1**. La versión de pandas sí influye: pandas 2.0+ introdujo soporte para backends Arrow nativos (usando `pd.ArrowDtype`), lo que reduce el gap de rendimiento con Polars para operaciones en columnas Arrow. pandas 3.x continúa esas mejoras. En versiones antiguas (1.x), la diferencia entre pandas y Polars era más pronunciada.

**13.**

Numba supera a NumPy en casos donde la operación tiene un loop explícito simple porque:
- Numba compila el loop exacto que necesita, sin generalidad. NumPy tiene funciones genéricas que manejan múltiples tipos y casos edge.
- Numba puede usar LLVM con optimizaciones agresivas específicas para el código escrito, incluyendo desenrollado de loops, vectorización SIMD automática, y eliminación de verificaciones de límites.
- NumPy en operaciones como `(X * params).sum(axis=1)` crea arrays intermedios temporales en memoria, mientras que el loop de Numba puede hacer la operación en registros del CPU sin materializar intermedios.

**14.**

Si se incluyera el costo de convertir de `pd.DataFrame` a `np.ndarray` o de `pd.DataFrame` a `pl.DataFrame`, las implementaciones NumPy, Numba y Polars serían penalizadas por ese costo adicional. La conversión de pandas a NumPy es generalmente rápida, pero la conversión a Polars implica copiar y convertir el formato Arrow, lo que puede tardar tanto como la predicción misma para datasets pequeños.

Los escenarios de producción en donde el costo no existiría son aquellos en los que los datos ya provienen en formato Arrow/Parquet, o si el modelo está integrado en un pipeline que opera siempre sobre NumPy arrays.

**15.**

Para 100 millones de filas en un servidor de producción **CPU**:
- Elegiriamos **Numba-JIT** como primera opción. Esta escala linealmente. Numba permite paralelizar el loop con `@numba.njit(parallel=True)` y `numba.prange`, aprovechando todos los cores sin GIL.

Si se dispusiera de una **GPU**:
- La elección cambiaría a **Numba CUDA** (`@numba.cuda.jit`). Una GPU moderna puede ejecutar esta multiplicación vectorial sobre 100M filas en segundos, ya que el dot product es una operación trivialmente paralelizable en GPU. El cuello de botella pasaría a ser la transferencia de datos desde CPU a GPU.

### 2.2 Entrenamiento y Comparación de `n_jobs` [0.5 Puntos]

Ahora entrenaremos un modelo más complejo: un **RandomForestRegressor** que usa las características de audio más una codificación del género musical para predecir `valence`. Compararemos el efecto de paralelizar el entrenamiento con el parámetro `n_jobs`.

**Código dado — pipeline encapsulado** (no modificar):

In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


def build_pipeline(n_jobs: int = 1) -> Pipeline:
    # En producción este pipeline usaría LGBMRegressor; aquí usamos RandomForest
    # para ilustrar el efecto de n_jobs de forma más pronunciada.
    return Pipeline(
        [
            (
                "column_transformer",
                ColumnTransformer(
                    [
                        ("ohe", OneHotEncoder(handle_unknown="ignore"), ["key", "mode", "genre"]),
                        (
                            "numerical",
                            "passthrough",
                            PARAM_COLS,
                        ),
                    ]
                ),
            ),
            ("random_forest", RandomForestRegressor(n_jobs=n_jobs, random_state=42)),
        ]
    )

In [14]:
# Entrena con n_jobs=1 y mide el tiempo
pipeline_1 = build_pipeline(n_jobs=1)
t0 = time.perf_counter()
pipeline_1.fit(X_train, y_train)
time_1job = time.perf_counter() - t0

# Entrena con n_jobs=-1 y mide el tiempo
pipeline_all = build_pipeline(n_jobs=-1)
t0 = time.perf_counter()
pipeline_all.fit(X_train, y_train)
time_all_jobs = time.perf_counter() - t0

# Calcula RMSE de ambos modelos
rmse_1 = root_mean_squared_error(y_test, pipeline_1.predict(X_test))
rmse_all = root_mean_squared_error(y_test, pipeline_all.predict(X_test))

print(f"n_jobs=1  → tiempo: {time_1job:.1f}s | RMSE: {rmse_1:.4f}")
print(f"n_jobs=-1 → tiempo: {time_all_jobs:.1f}s | RMSE: {rmse_all:.4f}")

n_jobs=1  → tiempo: 185.1s | RMSE: 0.1652
n_jobs=-1 → tiempo: 33.0s | RMSE: 0.1652


In [15]:
# Gráficos de tiempos y RMSE
df_perf = pd.DataFrame(
    {
        "configuracion": ["n_jobs=1", "n_jobs=-1"],
        "tiempo_s": [time_1job, time_all_jobs],
        "rmse": [rmse_1, rmse_all],
    }
)

px.bar(
    df_perf,
    x="configuracion",
    y="tiempo_s",
    title="Tiempo de entrenamiento según n_jobs",
    labels={"tiempo_s": "Tiempo (s)", "configuracion": "Configuración"},
    text_auto=".1f",
).show()

px.bar(
    df_perf,
    x="configuracion",
    y="rmse",
    title="RMSE según n_jobs",
    labels={"rmse": "RMSE", "configuracion": "Configuración"},
    text_auto=".4f",
).show()

### Preguntas [0.5 Puntos]

1. ¿Qué hace el parámetro `n_jobs` en RandomForest (y en general en scikit-learn)?
2. **¿Por qué aquí sí funciona el paralelismo real sin el problema del GIL?** (Pista: RandomForest en scikit-learn usa joblib con backend de procesos o threads nativos.)
3. ¿Cuánto mejoró el tiempo con `n_jobs=-1`? 
4. ¿Fue proporcional al número de CPUs disponibles en tu máquina? ¿Por qué no?
5. ¿Hubo diferencia en RMSE entre ambas versiones? ¿Era esperable? ¿Por qué?

**Respuestas:**

**1.**

El parámetro `n_jobs` controla el número de trabajadores paralelos a usar para el entrenamiento y predicción:
- `n_jobs=1`: Entrena en serie, un árbol a la vez.
- `n_jobs=k`: Usa k procesos/threads paralelos para construir los árboles.
- `n_jobs=-1`: Usa todos los núcleos disponibles del sistema.

En scikit-learn en general, `n_jobs` aplica a estimadores que son paralelizables.

**2.**

RandomForest en scikit-learn usa **joblib** como backend de paralelización. Joblib detecta automáticamente la mejor estrategia y para operaciones CPU-bound como entrenar árboles, usa el backend **loky** por defecto, que crea procesos separados. Cada proceso tiene su propio intérprete Python con su propio GIL, por lo que múltiples procesos pueden ejecutar código Python puro simultáneamente sin interferirse. Adicionalmente, el código interno de construcción de árboles en scikit-learn está implementado en Cython/C, que libera el GIL durante la ejecución. Esto permite que incluso con threads se obtuviera paralelismo real para la parte C del entrenamiento.

**3.**

Con `n_jobs=-1` se observa una reducción significativa en el tiempo de entrenamiento respecto a `n_jobs=1`. Ya que se pasa de 185.1s para `n_jobs=1` a 33.0s para `n_jobs=-1`. Mejoró alrededor de 6 veces.

**4.**

No fue exactamente proporcional, ya que en la máquina que se corrio se tienen 8 nucleos y la mejora fue de 6 veces. Si la máquina tiene N cores, el speedup teórico máximo sería N×, pero en la práctica es menor, porque crear N procesos, serializar los datos para cada proceso y recopilar resultados tiene un costo fijo. También tenemos que los árboles entrenados se combinan en el proceso principal del RandomForest, por lo que no es del todo paralelo, siendo un proceso serial.

**5.**

No hubo diferencia en RMSE entre `n_jobs=1` y `n_jobs=-1`. Esto es completamente esperable porque `n_jobs` solo controla cómo se distribuye el trabajo de entrenamiento entre workers, pero no cambia el algoritmo ni los datos. Cada árbol del RandomForest se entrena de forma idéntica independientemente de cuántos procesos se usen, siempre que el `random_state` sea el mismo. El resultado final es idéntico entre ambas configuraciones, por lo que el RMSE sobre el conjunto de test es el mismo.

# 3. Orquestación del Pipeline con Apache Airflow

En producción, los pipelines de datos y ML rara vez se ejecutan a mano desde un notebook. Se necesita:
- **Automatización**: que el pipeline corra periódicamente (diariamente, por hora…).
- **Dependencias**: que el entrenamiento solo comience si la carga de datos terminó exitosamente.
- **Monitoreo y reintentos**: que si una tarea falla, el sistema lo registre y reintente.

**Apache Airflow** resuelve exactamente esto. Define pipelines como **DAGs** (*Directed Acyclic Graphs*), donde cada nodo es una **tarea** y las aristas definen dependencias.

| Concepto | Descripción |
|----------|-------------|
| **DAG** | Grafo Dirigido Acíclico que representa el pipeline completo |
| **Operator** | Unidad de trabajo (`PythonOperator`, `BashOperator`, …) |
| **Task** | Instancia de un Operator dentro de un DAG |
| **XCom** | Mecanismo para pasar datos pequeños entre tareas |
| **schedule** | Expresión cron que indica cuándo ejecutar el DAG |

### Setup local


En la carpeta del Lab:

```bash
export AIRFLOW_HOME=$(pwd)
airflow db migrate          # inicializa la base de datos de metadata
# Ver la contraseña. Si no se en un comienzo, ejecutar airflow standalone, parar el proceso y luego ejecutar nuevamente este comando. 
cat $AIRFLOW_HOME/simple_auth_manager_passwords.json.generated 
airflow standalone       # levanta scheduler + webserver en http://localhost:8080
```

Los DAGs deben guardarse en `./dags`.

## 3.1 Implementación del DAG

**TO-DO [0.8 Puntos]:**
- [X] Implementar `task_load_data_fn`: cargar 5 batches en paralelo, guardar en disco como Parquet y pasar la ruta a la siguiente tarea usando XCom.
- [X] Implementar `task_train_model_fn`: recuperar la ruta de XCom, cargar el DataFrame, preparar X e y, entrenar `build_pipeline(n_jobs=-1)` e imprimir el tiempo.
- [X] Definir la dependencia entre tareas (`load_data >> train_model`).

El siguiente bloque es el template que debes completar en tu celda de respuesta.

In [ ]:
%%writefile ~/airflow/dags/spotify_pipeline_dag.py

from pathlib import Path
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

DATA_DIR = Path("/RUTA/ABSOLUTA/A/Labs/Lab9_v2/data")  # AJUSTA esta ruta
OUTPUT_PATH = Path("/tmp/spotify_data.parquet")

PARAM_COLS = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "tempo",
    "duration_ms",
    "year",
]

# ── Funciones auxiliares (dadas) ─────────────────────────────────────────────

def load_batch(path: str) -> pd.DataFrame:
    return pd.read_parquet(path)

def load_all_parallel(data_dir: Path, n_batches: int = 5) -> pd.DataFrame:
    paths = sorted(data_dir.glob("*.parquet"))[:n_batches]
    with ThreadPoolExecutor(max_workers=None) as executor:
        dfs = list(executor.map(load_batch, [str(p) for p in paths]))
    return pd.concat(dfs, ignore_index=True)

def build_pipeline(n_jobs: int = -1) -> Pipeline:
    return Pipeline(
        [
            (
                "column_transformer",
                ColumnTransformer(
                    [
                        ("ohe", OneHotEncoder(handle_unknown="ignore"), ["key", "mode", "genre"]),
                        ("numerical", "passthrough", PARAM_COLS),
                    ]
                ),
            ),
            ("random_forest", RandomForestRegressor(n_jobs=n_jobs, random_state=42)),
        ]
    )

# ── Funciones de las tareas de Airflow ───────────────────────────────────────

def task_load_data_fn(**context):
    """
    Carga 5 batches de datos en paralelo y guarda el resultado en disco.
    TODO: implementa esta función.
    - Usa load_all_parallel para cargar los datos.
    - Guarda el DataFrame resultante en OUTPUT_PATH (formato parquet).
    - Usa XCom para pasar la ruta del archivo a la siguiente tarea.
    """
    ...

def task_train_model_fn(**context):
    """
    Carga los datos desde disco y entrena el pipeline.
    TODO: implementa esta función.
    - Recupera la ruta del archivo desde XCom.
    - Lee el DataFrame desde esa ruta.
    - Prepara X e y, realiza el split 80/20.
    - Entrena build_pipeline(n_jobs=-1).
    - Imprime el tiempo de entrenamiento.
    """
    ...

# ── Definición del DAG ────────────────────────────────────────────────────────

with DAG(
    dag_id="spotify_pipeline",
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False,
    tags=["mds7202", "spotify"],
) as dag:
    load_data = PythonOperator(
        task_id="load_data",
        python_callable=task_load_data_fn,
    )

    train_model = PythonOperator(
        task_id="train_model",
        python_callable=task_train_model_fn,
    )

    # TODO: define la dependencia entre tareas (load_data debe ejecutarse antes que train_model)
    ...

In [ ]:
%%writefile ~/airflow/dags/spotify_pipeline_dag.py

from pathlib import Path
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

DATA_DIR = Path("/home/vixo/universidad/13vosemestre/lab-ciencia-de-datos/Laboratorios-MDS7202/labs/lab_9/data")
OUTPUT_PATH = Path("/tmp/spotify_data.parquet")

PARAM_COLS = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "tempo",
    "duration_ms",
    "year",
]

# ── Funciones auxiliares (dadas) ─────────────────────────────────────────────

def load_batch(path: str) -> pd.DataFrame:
    return pd.read_parquet(path)

def load_all_parallel(data_dir: Path, n_batches: int = 5) -> pd.DataFrame:
    paths = sorted(data_dir.glob("*.parquet"))[:n_batches]
    with ThreadPoolExecutor(max_workers=None) as executor:
        dfs = list(executor.map(load_batch, [str(p) for p in paths]))
    return pd.concat(dfs, ignore_index=True)

def build_pipeline(n_jobs: int = -1) -> Pipeline:
    return Pipeline(
        [
            (
                "column_transformer",
                ColumnTransformer(
                    [
                        ("ohe", OneHotEncoder(handle_unknown="ignore"), ["key", "mode", "genre"]),
                        ("numerical", "passthrough", PARAM_COLS),
                    ]
                ),
            ),
            ("random_forest", RandomForestRegressor(n_jobs=n_jobs, random_state=42)),
        ]
    )

# ── Funciones de las tareas de Airflow ───────────────────────────────────────

def task_load_data_fn(**context):
    """
    Carga 5 batches de datos en paralelo y guarda el resultado en disco.
    """
    df = load_all_parallel(DATA_DIR, n_batches=5)
    print(f"Datos cargados: {df.shape[0]} filas, {df.shape[1]} columnas")
    df.to_parquet(str(OUTPUT_PATH))
    print(f"DataFrame guardado en {OUTPUT_PATH}")
    context["ti"].xcom_push(key="data_path", value=str(OUTPUT_PATH))
    print(f"XCom push: data_path={OUTPUT_PATH}")

def task_train_model_fn(**context):
    """
    Carga los datos desde disco y entrena el pipeline.
    """
    data_path = context["ti"].xcom_pull(task_ids="load_data", key="data_path")
    print(f"Ruta recibida por XCom: {data_path}")
    df = pd.read_parquet(data_path)
    print(f"DataFrame cargado: {df.shape[0]} filas")
    X = df[PARAM_COLS + ["key", "mode", "genre"]]
    y = df["valence"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Split: {X_train.shape[0]} train / {X_test.shape[0]} test")
    pipeline = build_pipeline(n_jobs=-1)
    t0 = time.perf_counter()
    pipeline.fit(X_train, y_train)
    elapsed = time.perf_counter() - t0
    print(f"Entrenamiento completado en {elapsed:.1f}s")

# ── Definición del DAG ────────────────────────────────────────────────────────

with DAG(
    dag_id="spotify_pipeline",
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False,
    tags=["mds7202", "spotify"],
) as dag:
    load_data = PythonOperator(
        task_id="load_data",
        python_callable=task_load_data_fn,
    )

    train_model = PythonOperator(
        task_id="train_model",
        python_callable=task_train_model_fn,
    )

    load_data >> train_model

Writing /home/vixo/universidad/13vosemestre/lab-ciencia-de-datos/Laboratorios-MDS7202/labs/lab_9/dags/spotify_pipeline_dag.py


Una vez guardado el archivo, ejecuta el DAG con:

```bash
airflow standalone
```


### Pega aquí el output de las steps del DAG

- Step 1: 
...


- Step 2:
...

### Preguntas [1.2 Puntos]

1. ¿Qué es un **DAG**? ¿Qué significa que sea *Directed* (dirigido) y *Acyclic* (acíclico)? ¿Por qué importa la propiedad acíclica en un pipeline de datos?
2. ¿Qué es **Apache Airflow**? ¿Para qué tipo de problemas está diseñado y cuál es su unidad mínima de trabajo?
3. ¿Qué son los **Operators**? ¿Qué diferencia hay entre `PythonOperator` y `BashOperator`? ¿Cuándo usarías cada uno?
4. ¿Qué es **XCom** en Airflow? ¿Cómo funciona internamente (¿dónde se almacena?)? ¿Por qué **no** es adecuado para pasar DataFrames grandes entre tareas?
5. ¿Qué alternativa concreta usaste para pasar el DataFrame entre `load_data` y `train_model`? ¿Cuál sería la alternativa recomendada en producción (S3, GCS, DVC…)?
6. ¿Qué es el parámetro `schedule` de un DAG? ¿Cómo lo configurarías para que corra todos los días a las 3 AM?
7. ¿Qué diferencia hay entre Airflow y otras herramientas como **Prefect**, **Dagster**, **Luigi**, **Kubeflow**? ¿Cuál es la principal crítica que se le hace a Airflow?
8. ¿Por qué conviene orquestar el pipeline en Airflow en vez de simplemente ejecutar un script Python end-to-end?
9. ¿Qué pasa si `load_data` falla a mitad de camino? ¿Airflow reintenta automáticamente? ¿Cómo controlarías el número máximo de reintentos?
10. ¿Qué ventaja tiene que las tareas estén separadas (carga y entrenamiento) vs. una sola tarea monolítica, desde el punto de vista de debugging y eficiencia?
11. ¿Cómo podemos alertar si es que algún paso falla? ¿O si la pipeline se ejecuta correctamente?
12. En un pipeline de producción real, ¿qué otras tareas añadirías al DAG?

**Respuestas:**

**1. ¿Qué es un DAG? ¿Qué significa que sea Directed y Acyclic? ¿Por qué importa la propiedad acíclica?**

Un **DAG** (*Directed Acyclic Graph* — Grafo Dirigido Acíclico) es una estructura matemática de nodos conectados por aristas con dirección (de A hacia B) y sin ciclos (no hay un camino que regrese al nodo de origen).

- **Directed** (Dirigido): las aristas tienen dirección, indicando que la tarea A debe completarse antes que la tarea B. La dirección define la dependencia.
- **Acyclic** (Acíclico): no existen ciclos, es decir, no hay manera de partir de un nodo y volver al mismo siguiendo las aristas. Esto garantiza que el grafo tiene un orden topológico bien definido.

La propiedad acíclica es **fundamental en un pipeline de datos** porque garantiza que existe un orden de ejecución válido. Un ciclo crearía una dependencia circular: A espera a B, B espera a A → el pipeline nunca podría empezar. Airflow valida que el DAG no tenga ciclos al parsearlo; si los hubiera, lanzaría un error.

---

**2. ¿Qué es Apache Airflow? ¿Para qué tipo de problemas está diseñado? ¿Cuál es su unidad mínima de trabajo?**

**Apache Airflow** es una plataforma de orquestación de workflows de código abierto, desarrollada originalmente por Airbnb (2014) y donada a Apache (2016). Permite definir, programar, monitorear y ejecutar pipelines de datos como DAGs escritos en Python.

Está diseñado para: pipelines ETL/ELT, orquestación de tareas de ML (carga → preprocesamiento → entrenamiento → evaluación → despliegue), automatización de reportes, sincronización entre sistemas, y cualquier proceso que necesite ejecutarse en un orden definido y de forma repetible.

Su **unidad mínima de trabajo** es la **Task** (tarea): una instancia concreta de un Operator ejecutada dentro de un DAG en un momento específico.

---

**3. ¿Qué son los Operators? ¿Diferencia entre PythonOperator y BashOperator?**

Los **Operators** son plantillas de tareas que definen qué tipo de trabajo se realiza. Cada Operator encapsula la lógica de un tipo específico de tarea:

- **`PythonOperator`**: ejecuta una función Python (`python_callable`). Es el más flexible y el más usado en pipelines de ML/data. Permite acceder al contexto de Airflow (XCom, variables, etc.) via `**context`.

- **`BashOperator`**: ejecuta un comando de shell (Bash). Útil para invocar scripts, comandos CLI externos, herramientas que no tienen wrapper Python (p.ej., `dbt run`, `gsutil cp`, `spark-submit`), o tareas sencillas de sistema de archivos.

Otros operators relevantes: `EmailOperator` (enviar emails), `S3ToRedshiftOperator` (mover datos entre AWS S3 y Redshift), `KubernetesPodOperator` (lanzar pods en Kubernetes), `HttpOperator` (hacer requests HTTP).

---

**4. ¿Qué es XCom en Airflow? ¿Cómo funciona internamente? ¿Por qué NO es adecuado para DataFrames grandes?**

**XCom** (*Cross-Communication*) es el mecanismo de Airflow para pasar datos pequeños entre tareas. Funciona así: una tarea hace `xcom_push(key, value)` y la siguiente hace `xcom_pull(task_ids, key)`.

Internamente, XCom **serializa el valor (por defecto con pickle o JSON) y lo almacena en la base de datos de metadata de Airflow** (SQLite, PostgreSQL, MySQL según la configuración). Cuando la siguiente tarea hace `xcom_pull`, Airflow recupera el valor de esa base de datos y lo deserializa.

Por eso **NO es adecuado para DataFrames grandes**: la base de datos de metadata no está diseñada para almacenar megabytes o gigabytes de datos. Cargar un DataFrame de 500 MB en XCom colapsaría la base de datos de metadata, degradaría el rendimiento del scheduler y podría corromper el sistema. XCom está pensado para valores pequeños: rutas de archivos, enteros, strings, pequeños diccionarios de métricas.

---

**5. ¿Qué alternativa concreta usaste para pasar el DataFrame? ¿Cuál sería la recomendada en producción?**

En este laboratorio, la alternativa utilizada fue **guardar el DataFrame en disco como archivo Parquet** (`/tmp/spotify_data.parquet`) y pasar la **ruta del archivo** (string) a través de XCom. La tarea siguiente lee el archivo desde esa ruta.

En producción, la alternativa recomendada es almacenar en un **sistema de almacenamiento distribuido y persistente**:
- **AWS S3** o **Google Cloud Storage (GCS)**: guardar como Parquet y pasar la URI (`s3://bucket/path/file.parquet`). Esto permite que distintas máquinas accedan al mismo archivo.
- **DVC** (*Data Version Control*): versionado de datasets, permite trackear qué datos se usaron en cada run.
- **Delta Lake** o **Iceberg**: tablas con historial de cambios y ACID transactions.

El patrón `guardar en storage + pasar URI por XCom` es el estándar en producción.

---

**6. ¿Qué es el parámetro `schedule` de un DAG? ¿Cómo ejecutarlo todos los días a las 3 AM?**

El parámetro `schedule` define cuándo Airflow ejecuta el DAG automáticamente. Acepta:
- `None`: el DAG se ejecuta solo manualmente (como en este lab).
- Una **expresión cron**: `"0 3 * * *"` significa "a las 3:00 AM todos los días".
- Strings predefinidos: `"@daily"`, `"@hourly"`, `"@weekly"`.
- Objetos `timedelta`: `timedelta(hours=6)` para ejecutar cada 6 horas.

Para ejecutar todos los días a las 3 AM:
```python
with DAG(
    dag_id="spotify_pipeline",
    schedule="0 3 * * *",  # minuto=0, hora=3, cualquier día/mes/día_semana
    ...
)
```

La expresión cron `"0 3 * * *"` significa: minuto 0, hora 3, cualquier día del mes, cualquier mes, cualquier día de la semana.

---

**7. ¿Qué diferencia hay entre Airflow y otras herramientas como Prefect, Dagster, Luigi, Kubeflow?**

| Herramienta | Característica principal | Principal crítica a Airflow |
|---|---|---|
| **Airflow** | Maduro, muy adoptado, rico ecosistema de operators | El DAG se define estáticamente en Python; es difícil parametrizar; la UI es compleja; configuración pesada |
| **Prefect** | DAGs dinámicos, mejor manejo de errores, cloud-native | Menos maduro que Airflow, menor ecosistema |
| **Dagster** | Orientado a assets (no tareas), tipado de datos, observabilidad integrada | Curva de aprendizaje mayor, cambio de paradigma |
| **Luigi** | Simple, sin servidor central necesario | Sin UI rica, limitado para orquestación compleja |
| **Kubeflow** | Nativo en Kubernetes, orientado a ML pipelines | Complejidad de Kubernetes; overhead para proyectos pequeños |

La **principal crítica a Airflow** es que los DAGs son estáticos (no pueden variar su estructura en tiempo de ejecución), el scheduler puede ser un cuello de botella, y la configuración inicial es compleja. Prefect y Dagster son alternativas modernas que abordan estas limitaciones.

---

**8. ¿Por qué conviene orquestar el pipeline en Airflow en vez de ejecutar un script Python end-to-end?**

Un script Python end-to-end ejecuta todo en secuencia sin capacidad de: retries automáticos si algo falla, monitoreo visual del estado de cada paso, alertas cuando algo se rompe, logs estructurados por tarea, historial de ejecuciones, paralelización de tareas independientes, ni programación automática. Si el script falla a la mitad, no hay forma de saber exactamente dónde ni de reanudar desde ahí.

Airflow ofrece: UI web para monitorear el estado, reintentos configurables por tarea, logs por tarea, dependencias claras, historial de ejecuciones (DAG runs), alertas por email/Slack en caso de fallo, y la capacidad de ejecutar solo las tareas fallidas sin repetir las exitosas.

---

**9. ¿Qué pasa si `load_data` falla a mitad de camino? ¿Airflow reintenta automáticamente?**

Si `load_data` falla, Airflow marca esa tarea como `failed` y, dependiendo de la configuración, puede: (a) no hacer nada (por defecto), (b) reintentar automáticamente si se configuró `retries > 0`.

Para controlar los reintentos, se configura en el `PythonOperator`:
```python
load_data = PythonOperator(
    task_id="load_data",
    python_callable=task_load_data_fn,
    retries=3,                    # máximo 3 reintentos
    retry_delay=timedelta(minutes=5),  # esperar 5 min entre reintentos
)
```

Si `load_data` falla y no se reintenta exitosamente, `train_model` **no se ejecuta** (respeta la dependencia). El DAG queda en estado `failed` y se puede ver exactamente qué tarea falló y el traceback completo en la UI.

---

**10. ¿Qué ventaja tiene separar las tareas (carga y entrenamiento) vs. una tarea monolítica?**

**Desde el punto de vista de debugging**: si el entrenamiento falla (por ejemplo, error de dtype o de hiperparámetros), no es necesario volver a ejecutar la carga de datos (que puede tardar varios minutos). Se puede limpiar el estado de solo la tarea fallida y reejecutarla. Esto reduce el ciclo de depuración enormemente.

**Desde el punto de vista de eficiencia**: las tareas pueden ejecutarse en workers distintos (con distinta memoria o CPUs). La tarea de carga puede correr en una máquina con mucho almacenamiento rápido, y la de entrenamiento en una máquina con muchos cores. También permite reutilizar los datos cargados en múltiples tareas de entrenamiento (p.ej., entrenar varios modelos en paralelo sobre los mismos datos).

---

**11. ¿Cómo podemos alertar si algún paso falla, o si la pipeline se ejecuta correctamente?**

Airflow ofrece varios mecanismos de alertas:

1. **Email**: configurar `email_on_failure=True` y `email_on_success=True` en el DAG o en cada tarea, con el parámetro `email=['destinatario@ejemplo.com']`.

2. **Callbacks**: parámetros `on_failure_callback` y `on_success_callback` en cada tarea o a nivel de DAG. Se pueden usar para enviar mensajes a Slack, PagerDuty, o cualquier sistema de notificación.

```python
def notify_slack(context):
    # Usar un webhook de Slack para enviar mensaje
    ...

train_model = PythonOperator(
    task_id="train_model",
    python_callable=task_train_model_fn,
    on_failure_callback=notify_slack,
)
```

3. **Airflow Providers**: existen providers oficiales para Slack (`SlackWebhookOperator`), PagerDuty, Datadog, etc.

---

**12. En un pipeline de producción real, ¿qué otras tareas añadirías al DAG?**

Un DAG de producción completo para un pipeline ML típicamente incluiría:

1. **`validate_data`**: verificar que los archivos Parquet tienen el esquema esperado, sin nulls en columnas críticas, distribuciones dentro de rangos normales (data drift detection).
2. **`feature_engineering`**: transformaciones más complejas (encoding, normalización, creación de features).
3. **`evaluate_model`**: calcular RMSE, MAE, R² en un conjunto de validación y comparar con el modelo en producción (champion/challenger).
4. **`register_model`**: registrar el modelo en MLflow (u otro Model Registry) con sus métricas y artefactos.
5. **`deploy_model`**: si el nuevo modelo supera al actual, desplegar automáticamente a producción (actualizar endpoint de la API).
6. **`notify_success`/`notify_failure`**: enviar resumen de métricas al equipo por Slack/email al finalizar.
7. **`cleanup`**: eliminar archivos temporales, liberar espacio en `/tmp`.

Opcionalmente: tareas de **backtesting** del modelo, **A/B testing** setup, y **monitoreo continuo** de drift en producción.


# Conclusión

Eso ha sido todo para el lab de hoy. Recuerda que el laboratorio tiene un plazo de entrega de una semana. Cualquier duda, no dudes en contactarnos por el foro de U-Cursos.

<p align="center">
  <img src="https://media.giphy.com/media/l0HlBO7eyXzSZkJri/giphy.gif" width="300">
</p>